# Projek Akhir Pemrosesan Teks

Kelompok 4 :
1. Bima Setia Sugiharto (24031554040)
2. Muhammad Geralldo Agatha Saputra (24031554091)
3. Moh. Rasya Al Khalifi (24031554132)

## Import Data

In [86]:
import pandas as pd
import re
import nltk
nltk.download("punkt")
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\USER040824\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [87]:
df = pd.read_csv("livechatyt.csv")
df.head()

,timestamp,username,message
0,2025-11-02 14:56:26,@markxushi,HALO MPL
1,2025-11-02 14:56:27,@IRVAN_24,gak sesuai jadwal
2,2025-11-02 14:56:27,@ZyyyElMahir,IT'S ONIC TIME
3,2025-11-02 14:56:31,@Sandi-or4rd,1556
4,2025-11-02 14:56:32,@am_I02,jam 17:00 paling lambat jam 7:00 malaem haha


In [88]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   timestamp  10000 non-null  object
 1   username   10000 non-null  object
 2   message    9999 non-null   object
dtypes: object(3)
memory usage: 234.5+ KB


## Data Pre-Processing

In [89]:
df['message'] = df['message'].astype(str).str.lower()
df["message"] = df["message"].replace("false", pd.NA)
df = df.dropna(subset=["message"])

In [90]:
df = df.drop_duplicates(subset=['message'])
df

,timestamp,username,message
0,2025-11-02 14:56:26,@markxushi,halo mpl
1,2025-11-02 14:56:27,@IRVAN_24,gak sesuai jadwal
2,2025-11-02 14:56:27,@ZyyyElMahir,it's onic time
3,2025-11-02 14:56:31,@Sandi-or4rd,1556
4,2025-11-02 14:56:32,@am_I02,jam 17:00 paling lambat jam 7:00 malaem haha
...,...,...,...
9987,2025-11-02 16:52:36,@ahmadrifai3360,jam berapa main?
9992,2025-11-02 16:52:39,@mariasama3870,anjay rame banget
9993,2025-11-02 16:52:39,@taaclover,bismillahirrahmanirrahim ya allah bisa yok bisa
9995,2025-11-02 16:52:40,@Zynko20,er er qi main jam berapa bang?


In [91]:
def cleaning(text):
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@[A-Za-z0-9_]+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'(.)\1+', r'\1\1', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['message_clean'] = df['message'].astype(str).str.lower()
df['message_clean'] = df['message_clean'].apply(cleaning)
df = df.drop(columns=['message'])
df

,timestamp,username,message_clean
0,2025-11-02 14:56:26,@markxushi,halo mpl
1,2025-11-02 14:56:27,@IRVAN_24,gak sesuai jadwal
2,2025-11-02 14:56:27,@ZyyyElMahir,it s onic time
3,2025-11-02 14:56:31,@Sandi-or4rd,
4,2025-11-02 14:56:32,@am_I02,jam paling lambat jam malaem haha
...,...,...,...
9987,2025-11-02 16:52:36,@ahmadrifai3360,jam berapa main
9992,2025-11-02 16:52:39,@mariasama3870,anjay rame banget
9993,2025-11-02 16:52:39,@taaclover,bismillahirrahmanirrahim ya allah bisa yok bisa
9995,2025-11-02 16:52:40,@Zynko20,er er qi main jam berapa bang


In [92]:
df.isna().sum()

timestamp        0
username         0
message_clean    0
dtype: int64

In [93]:
norm = {
    "gg": "good game", "ggwp": "good game well played", "wp": "well played",
    "ml": "mobile legends", "mlbb": "mobile legends", "onicc": "onic",
    "gak": "tidak", "ga": "tidak", "nggak": "tidak", "gk": "tidak",
    "bgt": "banget", "wkwk": "tertawa", "wk": "tertawa", "awok": "tertawa",
    "pls": "tolong", "plis": "tolong", "ae": "alter ego",
    "utk": "untuk", "dgn": "dengan", "jg": "juga", "sy": "saya",
    "klo": "kalau", "krn": "karena", "tdk": "tidak", "sdh": "sudah"
}

In [94]:
def normalisasi(text):
    for i in norm:
        text = text.replace(i, norm[i])
    return text
df['message_clean'] = df['message_clean'].apply(lambda x: normalisasi(x))
df

,timestamp,username,message_clean
0,2025-11-02 14:56:26,@markxushi,halo mpl
1,2025-11-02 14:56:27,@IRVAN_24,tidak sesuai jadwal
2,2025-11-02 14:56:27,@ZyyyElMahir,it s onic time
3,2025-11-02 14:56:31,@Sandi-or4rd,
4,2025-11-02 14:56:32,@am_I02,jam paling lambat jam malalter egom haha
...,...,...,...
9987,2025-11-02 16:52:36,@ahmadrifai3360,jam berapa main
9992,2025-11-02 16:52:39,@mariasama3870,anjay rame banget
9993,2025-11-02 16:52:39,@taaclover,bismillahirrahmanirrahim ya allah bisa yok bisa
9995,2025-11-02 16:52:40,@Zynko20,er er qi main jam berapa bang


In [95]:
factory = StopWordRemoverFactory()
stopwords_list = factory.get_stop_words()
kata_penting = ['tidak', 'enggak', 'bukan', 'jangan', 'tapi', 'sangat', 'kurang', 'lebih', 'belum']
stopword = [word for word in stopwords_list if word not in kata_penting]
stopword.extend(['rt', 'user', 'url', 'yg', 'dg', 'dgn', 'ny', 'd', 'kalo', 'klo'])
topwords = set(stopword)

def hapus_stopword(text):
    words = text.split()
    return ' '.join([word for word in words if word not in stopword])
df['message_clean'] = df['message_clean'].apply(hapus_stopword)
df = df.drop(columns=['username', 'timestamp'])
df.to_csv("data_bersih.csv", index=False)

In [96]:
#tokenize
tokenized = df['message_clean'].apply(lambda x:x.split())
tokenized.head()

0                                         [halo, mpl]
1                             [tidak, sesuai, jadwal]
2                                 [it, s, onic, time]
3                                                  []
4    [jam, paling, lambat, jam, malalter, egom, haha]
Name: message_clean, dtype: object

In [97]:
# stemming
def stemming(text):
    factory = StemmerFactory()
    stemmer = factory.create_stemmer()
    hasil = []
    for i in text:
        data = stemmer.stem(i)
        hasil.append(data)
    data_clean = []
    data_clean = " ".join(hasil)
    print(data_clean)
    return data_clean
tokenized = tokenized.apply(stemming)
tokenized.to_csv('hasil_stemming.csv', index=False)

halo mpl
tidak sesuai jadwal
it s onic time

jam paling lambat jam malalter egom haha
jantidakn bahas tidak
rolling on the floor laughing rolling on the floor laughing rolling on the floor laughing
orang jadwal nya aja

ntidakret
lah jam
mpl id scamming
kobtoll
rrq malter egon habis subuh
woi lama
kasi evos kalah ama king alter ego
tidur aja dulu gess jam
its onic time
its onic time nauseated face nauseated face nauseated face nauseated face
mpl wacana cuki
woi duit bukan
lah kata
onic
hallo
pembohonk publik

mpl ntidakntuk
udh jam bal
jam malam kont
akun uda mulai live
mulai wehh lama kali
kalian nungood tidakmeu apa jam kocak
noob vs noob
stress tidakush bikin jadwal jam sgini akhir nya malter egonn nya malemm mahh
undur kocak

nya jam
guys
palintidakn antar jam
udah ribu nungood tidakmeuin loh loudly crying face loudly crying face
al in onic
woi manawoi
onic
lah gw kira telat
kocak udah hampir jam
kapan we
admin tll
jam pas kek
udh ki
sih kek kemarin
mana final oii
jam brp
hadehh
ph